# Argus prototype data 3: light curves

This notebook turns the target's minipix into a calibrated light curve:
aperture photometry with [photutils](https://photutils.readthedocs.io),
zeropoints from the photocal records, DQ masking, differential photometry
against comparison stars, and a period search. Notebook 2 introduces the
minipixes and their records.

## Parameters

Change these inputs first; leave the calculation cells intact.

In [ ]:
# The target: any star in the staged segment works (see notebook 2 for the list)
TARGET = dict(
    name="Gaia DR3 240827852899491328",
    ra=47.786654, dec=43.945067,  # deg, ICRS
    G=12.727, bp_rp=0.919,        # Gaia DR3 photometry
    gaia_frequency=1.2673242,     # 1/d, gaiadr3.vari_eclipsing_binary
)
SEG = 391749                      # the staged segment holding it

## Setup

In [ ]:
import os
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from astropy.visualization import AsinhStretch, ImageNormalize, ZScaleInterval

import argex
from argex import sky, times

warnings.filterwarnings("ignore", category=RuntimeWarning)  # all-NaN slices
GREEK = {"beta": "β", "rho": "ρ"}  # band names in plot labels
plt.rcParams.update({"figure.dpi": 110, "image.origin": "lower"})

# The staged sample. Set ARGUS_SAMPLE to read another copy, such as a download.
SAMPLE = Path(os.environ.get("ARGUS_SAMPLE", "~/s3/schmidt-observatory-system/argus/prototype_v1")).expanduser()
assert (SAMPLE / "manifest.json").is_file(), f"no sample at {SAMPLE}: set ARGUS_SAMPLE"

ra, dec = TARGET["ra"], TARGET["dec"]
FIELD = SAMPLE / f"seg_{SEG}"

px, py = sky.radec_to_pixel(SEG, ra, dec)  # the target's segment pixel
series = argex.read(FIELD / "minipix" / f"minipix_{sky.minipix_index(px, py):04d}.fits")
x, y = series.radec_to_pixel(ra, dec)       # ... and its pixel in the minipix
gr = (TARGET["bp_rp"] - 0.355) / 1.198      # g - r for the color term (notebook 02)

def night_of(mjd):
    # SRO local noon is about 19:00 UTC: one id per observing night
    return np.floor(np.asarray(mjd) - 0.79).astype(int)

## 1. Comparison stars

Two of the steps ahead need stars that do not vary: choosing the photometric
aperture, and differential photometry. They come from other active minipixes
of the same segment. We take nearby ones whose star sits well inside the
cutout, stays below saturation and is present in most frames, with no DQ
flags near it.

In [ ]:
grid = 64
active = sorted(int(p.stem.split("_")[1]) for p in (FIELD / "minipix").glob("minipix_*.fits"))


def star_position(s):
    # brightest pixel of the median cutout, refined by a centroid
    med = np.nanmedian(s.data, axis=0)
    py, px = np.unravel_index(np.nanargmax(med), med.shape)
    yy, xx = np.mgrid[:32, :32]
    w = np.clip(med - np.nanmedian(med), 0, None) * ((xx - px) ** 2 + (yy - py) ** 2 <= 9)
    return (w * xx).sum() / w.sum(), (w * yy).sum() / w.sum(), np.nanmax(med)


def grid_distance(a, b):
    return np.hypot(a % grid - b % grid, a // grid - b // grid)


comps = []
for idx in sorted(active, key=lambda i: grid_distance(i, series.idx)):
    if idx == series.idx:
        continue
    s = argex.read(FIELD / "minipix" / f"minipix_{idx:04d}.fits")
    cx, cy, peak = star_position(s)
    if not (8 <= cx <= 23 and 8 <= cy <= 23) or peak > 110 or len(s) < 0.8 * len(series):
        continue
    if s.dq(cx, cy, 7).any():
        continue
    comps.append(dict(idx=idx, series=s, x=cx, y=cy, peak=peak))
    if len(comps) == 15:
        break
print(len(comps), "comparison stars:", [c["idx"] for c in comps])

## 2. Aperture photometry with photutils

A circular aperture for the source and an annulus for the background. 

In [ ]:
from astropy.stats import SigmaClip
from astropy.table import Table
from photutils.aperture import ApertureStats, CircularAnnulus, CircularAperture, aperture_photometry

EXPTIME = 60.0                                # s, every frame
SEGMENT_PIXEL, DETECTOR_PIXEL = 0.879, 1.012  # arcsec
R = 4 * DETECTOR_PIXEL / SEGMENT_PIXEL        # the zeropoints' 4.05" aperture, in segment pixels
ANNULUS = (R + 3, 15.5)                       # what fits in a 32-pixel cutout


def aperture_flux(cube, x, y, radius=R, annulus=ANNULUS):
    # Background-subtracted flux (e-/s) and its error in each plane of a cube,
    # with the background the sigma-clipped median of the annulus.
    aperture = CircularAperture((x, y), r=radius)
    ring = CircularAnnulus((x, y), r_in=annulus[0], r_out=annulus[1])
    clip = SigmaClip(sigma=3.0)
    flux, err = np.full(len(cube), np.nan), np.full(len(cube), np.nan)
    for k, plane in enumerate(np.asarray(cube, np.float64)):
        sky_ = ApertureStats(plane, ring, sigma_clip=clip)
        phot = aperture_photometry(plane - sky_.median, aperture,
                                   error=np.full(plane.shape, sky_.std), method="exact")
        flux[k] = phot["aperture_sum"][0]
        # the sky noise from the annulus, plus the source's photon noise
        err[k] = np.hypot(phot["aperture_sum_err"][0], np.sqrt(max(flux[k], 0) / EXPTIME))
    return flux, err


def light_curve(frames, epochs, flux, err, zp, flags=None):
    # A table of calibrated magnitudes; a frame is good when it is calibrated,
    # has a positive flux and (with DQ flags) none of argex.dq.BAD under the aperture.
    mag = zp - 2.5 * np.log10(flux)
    good = np.isfinite(mag)
    if flags is not None:
        good &= (flags & int(argex.dq.BAD)) == 0
    return Table({
        "epoch": epochs, "mjd": times.to_mjd(epochs),
        "frame_key": [f.frame_key for f in frames],
        "flux": flux, "flux_err": err, "zp": zp,
        "mag": mag, "mag_err": 2.5 / np.log(10) * err / flux,
        "dq": np.zeros(len(flux), np.uint32) if flags is None else flags, "good": good,
    })


def minipix_light_curve(s, x, y, gr=None, radius=R):
    # Calibrated light curve of the star at cutout pixel (x, y) of a MinipixSeries,
    # with its own photocal and DQ records.
    flux, err = aperture_flux(s.data, x, y, radius)
    return light_curve(s.frames, s.epochs, flux, err,
                       zp=s.zeropoint(x, y, gr=gr), flags=s.dq(x, y, radius + 0.5))


print(f"aperture radius {R:.2f} px; annulus {ANNULUS[0]:.2f}-{ANNULUS[1]} px")

The error is the background noise in the aperture plus the source's photon
noise; it does not include the zeropoint's own uncertainty, and it is a
little optimistic, since interpolation to the segment plane correlates neighboring pixel.

Does 4.05″ suit these images? Measuring the brightest comparison star with a
range of radii shows the curve of growth (how much of the flux each radius
holds) and the scatter of its magnitudes over all frames and between
nightly medians, which should be small for a constant star.

In [ ]:
ref = max(comps, key=lambda c: c["peak"])
ref_rho = ref["series"].select(band="rho")
radii = [2, 3, 4, R, 6, 7, 8]
med_flux, rms, spread = [], [], []
for r in radii:
    lc = minipix_light_curve(ref_rho, ref["x"], ref["y"], radius=r)
    g = lc[lc["good"]]
    nights = night_of(g["mjd"])
    med_flux.append(np.median(g["flux"]))
    rms.append(np.std(g["mag"]))
    spread.append(np.std([np.median(g["mag"][nights == k]) for k in np.unique(nights)]))

fig, axes = plt.subplots(1, 2, figsize=(9, 3))
axes[0].plot(radii, np.array(med_flux) / med_flux[-1], "o-")
axes[0].set(xlabel="aperture radius (px)", ylabel="flux / flux(r=8)", title="curve of growth")
axes[1].plot(radii, 1e3 * np.array(rms), "o-", label="frame to frame")
axes[1].plot(radii, 1e3 * np.array(spread), "s--", label="night to night")
axes[1].set(xlabel="aperture radius (px)", ylabel="scatter (mmag)", title=f"minipix {ref['idx']}, ρ")
for ax in axes:
    ax.axvline(R, color="0.6", lw=0.8, ls=":")
axes[1].legend()
plt.tight_layout(); plt.show()

The dotted line is the zeropoints' radius, and the scatter drops sharply
there. Each frame's zeropoint was fitted to 4.05″ apertures in that frame,
so it already absorbs that frame's aperture loss, which changes with the
seeing. At any other radius, the difference between the two apertures
varies from frame to frame and shows up as scatter. 

## 3. The calibrated light curve

Now the target, in both bands:

In [ ]:
lcs = {b: minipix_light_curve(series.select(band=b), x, y, gr=gr) for b in ("beta", "rho")}
lcs["rho"][:5]

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.5))
for band, color in (("beta", "tab:blue"), ("rho", "tab:red")):
    lc = lcs[band]
    g = lc["good"]
    ax.errorbar(lc["mjd"][g], lc["mag"][g], lc["mag_err"][g], fmt=".", ms=2, lw=0.5, color=color,
                label=f"{GREEK[band]}: {g.sum()} good of {len(lc)} frames")
ax.invert_yaxis()
ax.set(xlabel="MJD (TT)", ylabel="AB mag", title=TARGET["name"])
ax.legend()
plt.show()

## 4. Differential photometry

Anything that shifts every star in a frame alike (a zeropoint error, thin
cloud, a seeing-dependent aperture loss) shows up on the comparison stars too,
and can be removed. Their frames line up with the target's by **frame key**.
The per-frame ensemble offset is the median over comparison stars of each
star's deviation from its own median magnitude; subtracting it gives the
differential light curve. The bottom panel shows the offsets: they are
small, because the per-frame calibration has already removed most of what is
common to the field. At the scale of the eclipses the calibrated points
(grey) mostly hide behind the differential points, as much of the frame-level correction is captured in the zeropoint fit.

In [ ]:
def differential(band):
    lc = lcs[band]
    keys = np.asarray(lc["frame_key"])
    deviations = []
    for comp in comps:
        c = minipix_light_curve(comp["series"].select(band=band), comp["x"], comp["y"])
        c = c[c["good"]]
        dev = dict(zip(np.asarray(c["frame_key"]), np.asarray(c["mag"] - np.median(c["mag"]))))
        deviations.append([dev.get(k, np.nan) for k in keys])
    deviations = np.array(deviations)
    # drop comparison stars that scatter far more than the rest (variables, blends)
    rms = np.nanstd(deviations - np.nanmedian(deviations, axis=0), axis=1)
    keep = rms < 2 * np.median(rms)
    offset = np.nanmedian(deviations[keep], axis=0)
    out = lc.copy()
    out["offset"] = offset
    out["mag_diff"] = lc["mag"] - offset
    out["good"] &= np.isfinite(offset)
    print(f"{band}: {keep.sum()} of {len(keep)} comparison stars kept")
    return out


dlcs = {b: differential(b) for b in ("beta", "rho")}

fig, axes = plt.subplots(3, 1, figsize=(9, 7), sharex=True, height_ratios=(2, 2, 1.3))
for ax, band, color in zip(axes, ("beta", "rho"), ("tab:blue", "tab:red")):
    lc = dlcs[band]; g = lc["good"]
    ax.plot(lc["mjd"][g], lc["mag"][g], ".", ms=2, color="0.7", label="calibrated")
    ax.plot(lc["mjd"][g], lc["mag_diff"][g], ".", ms=2, color=color, label="differential")
    ax.invert_yaxis(); ax.set_ylabel(f"{GREEK[band]} (AB mag)")
    ax.legend(fontsize=8, markerscale=4, loc="upper right")
    axes[2].plot(lc["mjd"][g], 1e3 * lc["offset"][g], ".", ms=2, color=color, label=GREEK[band])
axes[2].axhline(0, color="0.5", lw=0.6)
axes[2].set(xlabel="MJD (TT)", ylabel="ensemble offset\n(mmag)")
axes[2].legend(fontsize=8, markerscale=4, loc="upper right")
plt.show()

## 5. Period search and phase folding

A Lomb–Scargle periodogram of the differential ρ light curve looks for the
variability period.

In [ ]:
from astropy.timeseries import LombScargle

lc = dlcs["rho"]; g = lc["good"]
t, m = np.asarray(lc["mjd"][g]), np.asarray(lc["mag_diff"][g])
freq, power = LombScargle(t, m).autopower(minimum_frequency=0.3, maximum_frequency=12,
                                          samples_per_peak=20)
f_peak = freq[np.argmax(power)]

fig, ax = plt.subplots(figsize=(9, 2.5))
ax.plot(freq, power, lw=0.7)
ax.axvline(f_peak, color="tab:orange", lw=1, ls="--", label=f"highest peak, {f_peak:.3f} /d")
ax.axvline(2 * TARGET["gaia_frequency"], color="k", lw=1, ls=":",
           label="2 × Gaia orbital frequency (see below)")
ax.set(xlabel="frequency (1/d)", ylabel="power")
ax.legend(fontsize=8, loc="upper right")
plt.show()
print(f"highest peak: {f_peak:.4f} /d (period {1 / f_peak:.5f} d)")

The periodogram is a comb of almost equally tall peaks spaced one cycle per
day apart. These are aliases: the star is only observed for a few hours
each night, at nearly the same times, so a frequency $f$ and $f \pm 1$ /d fit
the data almost equally well. Eleven nights from the past month cannot settle which one is real
on their own.

Gaia DR3 measured this binary's orbit independently
(`gaiadr3.vari_eclipsing_binary`, which you can query with
`astroquery.gaia`). Its orbital frequency is half our highest peak: an
eclipsing binary has two eclipses per orbit, so its light curve repeats twice
per orbit.

In [ ]:
f_gaia = TARGET["gaia_frequency"]
print(f"Gaia orbital period {1 / f_gaia:.5f} d; 2 x our peak period {2 / f_peak:.5f} d"
      f" ({(2 / f_peak) / (1 / f_gaia) - 1:+.2%})")
period = 2 / f_peak

Folding both bands at the orbital period shows the two eclipses. In β, which
covers both, they differ in depth (as Gaia's own eclipse depths do), because
the two stars differ in surface brightness.

In [ ]:
# phase zero at the deepest point of the folded rho curve
phase = (t / period) % 1
bins = np.linspace(0, 1, 51)
binned = [np.median(m[(phase >= a) & (phase < b)]) for a, b in zip(bins[:-1], bins[1:])]
t0 = (bins[np.nanargmax(binned)] + 0.01) * period

fig, ax = plt.subplots(figsize=(8, 4))
for band, color in (("beta", "tab:blue"), ("rho", "tab:red")):
    lc = dlcs[band]; g = lc["good"]
    ph = ((np.asarray(lc["mjd"][g]) - t0) / period) % 1
    mag = np.asarray(lc["mag_diff"][g])
    for k in (0, 1):
        ax.plot(ph + k, mag, ".", ms=2, color=color, label=GREEK[band] if k == 0 else None)
ax.invert_yaxis()
ax.set(xlabel=f"phase (P = {period:.5f} d), shown twice", ylabel="differential AB mag",
       title=TARGET["name"])
ax.legend(markerscale=4)
plt.show()

## 6. Saving the results

The light curves are astropy tables; ECSV keeps their units and metadata.

In [ ]:
for band in ("beta", "rho"):
    dlcs[band].write(f"lightcurve_{band}.ecsv", overwrite=True)
print(dlcs["rho"].colnames)

## Notes

* **Photometric System** AB magnitudes in each camera's band (β, ρ). The photocal records carry no zeropoint
  uncertainty, so `mag_err` is the flux error alone.
* **Apertures.** The zeropoints hold for fluxes in e⁻/s in a 4.05″
  aperture. At other radii, the magnitudes will shift by the curve of growth
  between the two apertures.
* **Try another star.** Any file in the sample's `minipix/` directory works
  the same way; its star's Gaia color sets `gr`.